# Решения: DP 2D: стоимость и число маршрутов по сетке

**Для преподавателя.** Полный эталон к `lesson.ipynb` и `homework.ipynb`; ученикам до сдачи не показывать.

In [ ]:
from pathlib import Path
import csv


def find_data(name):
    import urllib.request
    for path in (
        Path(name),
        Path("../") / name,
        Path("../../data") / name,
        Path("../data") / name,
        Path("../../../data") / name,
    ):
        if path.exists():
            return path.resolve()
    url = (
        "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/"
        "modules/08_09_courier_dp/data/" + name
    )
    dest = Path(name)
    urllib.request.urlretrieve(url, dest)
    return dest.resolve()


def load_coin_cases():
    rows = []
    with find_data("coin_change_cases.csv").open(encoding="utf-8") as file:
        for row in csv.DictReader(file):
            rows.append((
                row["case_id"],
                int(row["amount"]),
                [int(value) for value in row["coins"].split()],
                int(row["expected_min_coins"]),
            ))
    return rows


def load_grid():
    with find_data("route_cost_grid_4x5.csv").open(encoding="utf-8") as file:
        reader = csv.reader(file)
        next(reader)
        return [[int(value) for value in row] for row in reader]


COIN_CASES = load_coin_cases()
ROUTE_GRID = load_grid()
assert len(COIN_CASES) == 5
assert len(ROUTE_GRID) == 4 and len(ROUTE_GRID[0]) == 5


## Урок. 1. Два параметра состояния

Создайте таблицу того же размера, что и сетка. Ячейка `(row, col)` будет хранить ответ для маршрута до этой точки.

In [ ]:
rows = len(ROUTE_GRID)
cols = len(ROUTE_GRID[0])
dp = [[0] * cols for _ in range(rows)]
assert len(dp) == rows
assert all(len(row) == cols for row in dp)
assert dp[0][0] == 0


## Урок. 2. Границы таблицы стоимости

В первую строку можно прийти только слева, в первый столбец — только сверху. Заполните эти базовые состояния.

In [ ]:
dp = [[0] * cols for _ in range(rows)]
dp[0][0] = ROUTE_GRID[0][0]
for col in range(1, cols):
    dp[0][col] = dp[0][col - 1] + ROUTE_GRID[0][col]
for row in range(1, rows):
    dp[row][0] = dp[row - 1][0] + ROUTE_GRID[row][0]
assert dp[0] == [1, 4, 5, 10, 11]
assert [dp[row][0] for row in range(rows)] == [1, 3, 8, 12]


## Урок. 3. Минимальная стоимость маршрута

Во внутреннюю ячейку можно прийти сверху или слева. Добавьте стоимость текущей клетки к меньшему из двух предыдущих ответов.

In [ ]:
def min_path_cost(grid):
    rows, cols = len(grid), len(grid[0])
    dp = [[0] * cols for _ in range(rows)]
    dp[0][0] = grid[0][0]
    for col in range(1, cols):
        dp[0][col] = dp[0][col - 1] + grid[0][col]
    for row in range(1, rows):
        dp[row][0] = dp[row - 1][0] + grid[row][0]
    for row in range(1, rows):
        for col in range(1, cols):
            dp[row][col] = grid[row][col] + min(dp[row - 1][col], dp[row][col - 1])
    return dp[-1][-1]


assert min_path_cost(ROUTE_GRID) == 11
assert min_path_cost([[7]]) == 7
assert min_path_cost([[1, 2], [3, 4]]) == 7


## Урок. 4. Вернуть всю таблицу

Инженеру нужна диагностика, а не только последняя ячейка. Реализуйте функцию, возвращающую таблицу минимальных стоимостей.

In [ ]:
def min_cost_table(grid):
    rows, cols = len(grid), len(grid[0])
    dp = [[0] * cols for _ in range(rows)]
    dp[0][0] = grid[0][0]
    for col in range(1, cols):
        dp[0][col] = dp[0][col - 1] + grid[0][col]
    for row in range(1, rows):
        dp[row][0] = dp[row - 1][0] + grid[row][0]
    for row in range(1, rows):
        for col in range(1, cols):
            dp[row][col] = grid[row][col] + min(dp[row - 1][col], dp[row][col - 1])
    return dp


costs = min_cost_table(ROUTE_GRID)
assert costs[-1][-1] == 11
assert costs[1][2] == 8
assert all(costs[row][col] >= ROUTE_GRID[row][col] for row in range(rows) for col in range(cols))


## Урок. 5. Восстановить маршрут

Идите от правого нижнего угла к соседу с меньшей накопленной стоимостью. Верните координаты от старта до финиша.

In [ ]:
def restore_min_path(grid):
    dp = min_cost_table(grid)
    row, col = len(grid) - 1, len(grid[0]) - 1
    path = [(row, col)]
    while row > 0 or col > 0:
        if row == 0:
            col -= 1
        elif col == 0:
            row -= 1
        elif dp[row - 1][col] <= dp[row][col - 1]:
            row -= 1
        else:
            col -= 1
        path.append((row, col))
    return list(reversed(path))


path = restore_min_path(ROUTE_GRID)
assert path[0] == (0, 0) and path[-1] == (3, 4)
assert len(path) == 8
assert sum(ROUTE_GRID[row][col] for row, col in path) == 11


## Урок. 6. Число маршрутов без препятствий

Теперь значение клетки — количество путей до неё. Стоимости игнорируются; переход складывает число путей сверху и слева.

In [ ]:
def count_paths(rows, cols):
    dp = [[0] * cols for _ in range(rows)]
    dp[0][0] = 1
    for row in range(rows):
        for col in range(cols):
            if row == 0 and col == 0:
                continue
            top = dp[row - 1][col] if row > 0 else 0
            left = dp[row][col - 1] if col > 0 else 0
            dp[row][col] = top + left
    return dp[-1][-1]


assert count_paths(4, 5) == 35
assert count_paths(1, 7) == 1
assert count_paths(2, 2) == 2


## Урок. 7. Запретные клетки

Запретная клетка получает ноль путей независимо от соседей. Проверьте отдельно случай заблокированного старта.

In [ ]:
def count_paths_with_blocks(rows, cols, blocks):
    dp = [[0] * cols for _ in range(rows)]
    if (0, 0) in blocks:
        return 0
    dp[0][0] = 1
    for row in range(rows):
        for col in range(cols):
            if (row, col) in blocks:
                dp[row][col] = 0
                continue
            if row == 0 and col == 0:
                continue
            top = dp[row - 1][col] if row > 0 else 0
            left = dp[row][col - 1] if col > 0 else 0
            dp[row][col] = top + left
    return dp[-1][-1]


assert count_paths_with_blocks(4, 5, {(1, 1), (2, 3)}) == 7
assert count_paths_with_blocks(2, 2, {(0, 0)}) == 0
assert count_paths_with_blocks(2, 2, set()) == 2


## Урок. 8. Эксперимент: чувствительность маршрута

Увеличивайте стоимость клетки `(2, 2)` от исходной до 20. Зафиксируйте значения, при которых оптимальный маршрут меняется.

In [ ]:
original_path = restore_min_path(ROUTE_GRID)
changes = []
for value in range(1, 21):
    changed = [row[:] for row in ROUTE_GRID]
    changed[2][2] = value
    if restore_min_path(changed) != original_path:
        changes.append(value)
PATH_NOTE = (
    "Таблица хранит лучший результат для каждого префикса маршрута. Изменение "
    "одной клетки меняет все состояния правее и ниже неё, поэтому обратный "
    "проход может выбрать другую цепочку координат."
)
assert changes
assert all(1 <= value <= 20 for value in changes)
assert len(PATH_NOTE) >= 100


## Урок. 9. Самостоятельно: максимальная стоимость

Перенесите тот же 2D-шаблон на критерий максимума. Верните число и один маршрут, не изменяя допустимые ходы.

In [ ]:
def max_path_with_route(grid):
    rows, cols = len(grid), len(grid[0])
    dp = [[0] * cols for _ in range(rows)]
    dp[0][0] = grid[0][0]
    for col in range(1, cols):
        dp[0][col] = dp[0][col - 1] + grid[0][col]
    for row in range(1, rows):
        dp[row][0] = dp[row - 1][0] + grid[row][0]
    for row in range(1, rows):
        for col in range(1, cols):
            dp[row][col] = grid[row][col] + max(dp[row - 1][col], dp[row][col - 1])
    row, col = rows - 1, cols - 1
    path = [(row, col)]
    while row > 0 or col > 0:
        if row == 0:
            col -= 1
        elif col == 0:
            row -= 1
        elif dp[row - 1][col] >= dp[row][col - 1]:
            row -= 1
        else:
            col -= 1
        path.append((row, col))
    return dp[-1][-1], list(reversed(path))


best, path = max_path_with_route(ROUTE_GRID)
assert best == 19
assert best == sum(ROUTE_GRID[row][col] for row, col in path)
assert path[0] == (0, 0) and path[-1] == (3, 4)
assert len(path) == 8


## ДЗ. A1. Маршруты с препятствиями

Реализуйте подсчёт для произвольного размера сетки.

In [ ]:
def blocked_routes(rows, cols, blocks):
    dp = [[0] * cols for _ in range(rows)]
    if (0, 0) in blocks:
        return 0
    dp[0][0] = 1
    for row in range(rows):
        for col in range(cols):
            if (row, col) in blocks:
                dp[row][col] = 0
            elif row != 0 or col != 0:
                dp[row][col] = (
                    (dp[row - 1][col] if row > 0 else 0)
                    + (dp[row][col - 1] if col > 0 else 0)
                )
    return dp[-1][-1]


assert blocked_routes(3, 4, {(1, 1)}) == 4
assert blocked_routes(1, 4, {(0, 2)}) == 0


## ДЗ. A2. Самый выгодный маршрут

Верните максимальную сумму для новой сетки.

In [ ]:
def max_path_sum(grid):
    rows, cols = len(grid), len(grid[0])
    dp = [[0] * cols for _ in range(rows)]
    dp[0][0] = grid[0][0]
    for row in range(rows):
        for col in range(cols):
            if row == 0 and col == 0:
                continue
            top = dp[row - 1][col] if row > 0 else -10**9
            left = dp[row][col - 1] if col > 0 else -10**9
            dp[row][col] = grid[row][col] + max(top, left)
    return dp[-1][-1]


grid = [[4, 1, 2], [7, 0, 3], [2, 8, 1]]
assert max_path_sum(grid) == 22
assert max_path_sum([[5]]) == 5


## ДЗ. A3. Маршрут минимальной стоимости

Верните координаты и проверьте стоимость по исходной сетке.

In [ ]:
def min_route(grid):
    rows, cols = len(grid), len(grid[0])
    dp = [[0] * cols for _ in range(rows)]
    dp[0][0] = grid[0][0]
    for row in range(rows):
        for col in range(cols):
            if row == 0 and col == 0:
                continue
            top = dp[row - 1][col] if row > 0 else 10**9
            left = dp[row][col - 1] if col > 0 else 10**9
            dp[row][col] = grid[row][col] + min(top, left)
    row, col = rows - 1, cols - 1
    route = [(row, col)]
    while row or col:
        if row > 0 and (col == 0 or dp[row - 1][col] <= dp[row][col - 1]):
            row -= 1
        else:
            col -= 1
        route.append((row, col))
    return list(reversed(route))


route = min_route(ROUTE_GRID)
assert route[0] == (0, 0) and route[-1] == (3, 4)
assert sum(ROUTE_GRID[row][col] for row, col in route) == 11
assert all((b[0] - a[0], b[1] - a[1]) in {(1, 0), (0, 1)} for a, b in zip(route, route[1:]))


## ДЗ. B1. Сколько оптимальных маршрутов

Верните минимальную стоимость и число маршрутов с такой стоимостью.

In [ ]:
def count_min_cost_paths(grid):
    rows, cols = len(grid), len(grid[0])
    cost = [[10**9] * cols for _ in range(rows)]
    count = [[0] * cols for _ in range(rows)]
    cost[0][0], count[0][0] = grid[0][0], 1
    for row in range(rows):
        for col in range(cols):
            if row == 0 and col == 0:
                continue
            candidates = []
            if row > 0:
                candidates.append((cost[row - 1][col], count[row - 1][col]))
            if col > 0:
                candidates.append((cost[row][col - 1], count[row][col - 1]))
            best_previous = min(value for value, _ in candidates)
            cost[row][col] = grid[row][col] + best_previous
            count[row][col] = sum(number for value, number in candidates if value == best_previous)
    return cost[-1][-1], count[-1][-1]


assert count_min_cost_paths([[1, 1], [1, 1]]) == (3, 2)
assert count_min_cost_paths(ROUTE_GRID) == (11, 1)
